# Clase 010 — OOP básico, dataclasses, herencia

**Parte 0** · Ramalho caps. 5 y 14.

> 🎯 Clases cuando aportan, `@dataclass` para records, herencia con criterio, dunders esenciales.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
from dataclasses import dataclass, field, FrozenInstanceError
from math import sqrt
from typing import NamedTuple

## 1️⃣ Clase mínima

```python
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def distancia_origen(self):
        return (self.x**2 + self.y**2) ** 0.5
```

Problema: sin `__repr__` ni `__eq__`, debugar es horrible (`<__main__.Punto object at 0x7f...>`).

In [ ]:
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f'Punto(x={self.x}, y={self.y})'
    def __eq__(self, other):
        return isinstance(other, Punto) and (self.x, self.y) == (other.x, other.y)
    def __add__(self, other):
        return Punto(self.x + other.x, self.y + other.y)
    def distancia_origen(self):
        return sqrt(self.x**2 + self.y**2)

p1 = Punto(3, 4)
p2 = Punto(3, 4)
print(p1)            # Punto(x=3, y=4)
print(p1 == p2)      # True
print(p1 + Punto(1, 1))   # Punto(x=4, y=5)
print(f'distancia: {p1.distancia_origen()}')

## 2️⃣ `@dataclass` — el atajo

Escribir `__init__`/`__repr__`/`__eq__` a mano para 10 atributos = error humano. `@dataclass` los genera.

In [ ]:
@dataclass
class Estudiante:
    nombre: str
    notas: list[float] = field(default_factory=list)

    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas) if self.notas else 0.0

estudiantes = [
    Estudiante('Ana', [6.5, 7.0, 5.8]),
    Estudiante('Bob', [4.2, 5.5]),
    Estudiante('Cris', [7.0, 6.8, 7.2, 6.5]),
]

for e in sorted(estudiantes, key=lambda x: x.promedio(), reverse=True):
    print(f'{e.nombre}: {e.promedio():.2f}')

**Nota**: `default_factory=list` evita el [bug del default mutable de la clase 006](../006-python-tipos-estructuras-control-de-flujo/README.md).

## 3️⃣ `frozen=True` — inmutabilidad

Ideal para records que viajan como datos puros (eventos, configs, coordenadas):

In [ ]:
@dataclass(frozen=True)
class Vector:
    x: float
    y: float

v = Vector(3.0, 4.0)
print(v)

try:
    v.x = 99.0
except FrozenInstanceError as e:
    print(f'Bien — no se puede mutar: {e}')

# Como bonus: frozen=True lo hace hashable, sirve como key de dict
origenes = {Vector(0, 0): 'origen', Vector(1, 0): 'eje x'}
print(origenes[Vector(0, 0)])

## 4️⃣ Herencia + `super()`

```python
class Animal:
    def __init__(self, nombre):
        self.nombre = nombre
    def hablar(self):
        return 'sonido genérico'

class Perro(Animal):
    def hablar(self):
        return 'guau'
```

`super()` invoca al método de la clase base — útil para extender, no reemplazar:

In [ ]:
class Animal:
    def __init__(self, nombre):
        self.nombre = nombre
    def hablar(self):
        return 'sonido genérico'
    def __repr__(self):
        return f'{type(self).__name__}({self.nombre!r})'

class Perro(Animal):
    def __init__(self, nombre, raza):
        super().__init__(nombre)
        self.raza = raza
    def hablar(self):
        return f'guau (soy {self.raza})'

class Gato(Animal):
    def hablar(self):
        return 'miau'

animales = [Perro('Rex', 'pastor'), Gato('Mishi'), Animal('???')]
for a in animales:
    print(f'{a}: {a.hablar()}')

## 5️⃣ Composición > herencia

La regla **"is-a vs has-a"**:

- `Perro` **is-a** `Animal` → herencia OK.
- `Coche` **has-a** `Motor` → composición (el coche tiene un motor, no es un motor).

Herencia mal usada acopla y crea jerarquías frágiles ("problema del diamante").

In [ ]:
# Composición: Coche contiene Motor
@dataclass
class Motor:
    potencia_hp: int
    def arrancar(self):
        return f'rrrrr ({self.potencia_hp} HP)'

@dataclass
class Coche:
    marca: str
    motor: Motor   # has-a, no is-a
    def arrancar(self):
        return f'{self.marca}: {self.motor.arrancar()}'

c = Coche('Toyota', Motor(180))
print(c.arrancar())

## 6️⃣ ¿Cuándo cada herramienta?

| Necesito… | Usa |
|---|---|
| Record inmutable, hashable, sin métodos | `NamedTuple` o `@dataclass(frozen=True)` |
| Record mutable con algunos métodos | `@dataclass` |
| Validación, computed fields, lifecycle | `pydantic.BaseModel` (verás en MLOps) |
| Estructura mutable sin comportamiento | `dict` o `TypedDict` |
| Lógica compleja, estado, polimorfismo | Clase normal con `__init__` |

## ✅ Checklist

- [ ] Sé escribir una clase con `__init__` y dunders
- [ ] Uso `@dataclass` en vez de boilerplate manual
- [ ] Entiendo cuándo `frozen=True` aporta
- [ ] Sé heredar y usar `super()`
- [ ] Prefiero composición salvo cuando is-a es genuino

## 📝 Homework

Ver `README.md`. `Punto`, `Estudiante`, `Vector` frozen, jerarquía `Animal`.

## 📖 Definiciones y características

**Clase / instancia**

Una **clase** es una plantilla (`class Punto:`); una **instancia** es un objeto concreto (`p = Punto(3, 4)`). `__init__` se llama al crear la instancia. `self` es la convención para referirse a la instancia dentro de los métodos.

**Método dunder ("magic method")**

Método con doble underscore (`__init__`, `__repr__`, `__eq__`, `__lt__`, `__len__`, `__iter__`, `__add__`). Python los invoca implícitamente con sintaxis especial (`len(obj)` → `obj.__len__()`).

**`@dataclass`**

Decorador que genera `__init__`, `__repr__`, `__eq__` automáticamente desde las anotaciones de tipo de la clase. Reduce boilerplate. Con `frozen=True` la hace inmutable y hashable.

**Herencia**

`class B(A)` — B hereda atributos y métodos de A; puede sobreescribirlos. `super()` invoca al método de la clase padre. Múltiple herencia existe pero se complica (MRO).

**Composición**

"B *tiene un* A" (atributo) en vez de "B *es un* A" (herencia). Generalmente preferible: menos acoplamiento, sin problemas de herencia múltiple/diamante.

**Polimorfismo**

Distintas clases responden al mismo método con comportamiento distinto (`Animal.hablar()` → 'guau' o 'miau' según la subclase). Permite tratar instancias heterogéneas uniformemente.

**NamedTuple vs dataclass vs TypedDict**

**NamedTuple**: tupla con nombres, inmutable, hashable, sin métodos custom. **dataclass**: clase con boilerplate auto, mutable por default (mejor con métodos). **TypedDict**: dict con esquema (estructural, no nominal).

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| Olvidé `self` en un método y el error es confuso | Cualquier método de instancia recibe `self` automáticamente. Sin él, Python lo confunde con otra cosa. **Fix**: siempre `def metodo(self, ...)`. |
| `@dataclass` con field mutable default rompe | `@dataclass class X: items: list = []` lanza `ValueError: mutable default ...`. **Fix**: `items: list = field(default_factory=list)`. |
| `__eq__` definido pero `__hash__` rompe | Definir `__eq__` sin `__hash__` hace la clase no-hashable automáticamente. **Fix**: define ambos, o usa `@dataclass(frozen=True)` (lo hace por ti). |
| `super().__init__(...)` olvidado en subclase | Atributos del padre quedan sin inicializar. **Fix**: si la subclase override `__init__`, llama `super().__init__(...)` explícitamente. |
| Modifico atributo y otra instancia también cambió | Asignaste un mutable como class attribute, no instance: `class X: items = []` — todas las instancias comparten esa lista. **Fix**: inicializa en `__init__` (`self.items = []`). |

## ❓ Preguntas frecuentes

**❓ ¿Cuándo necesito OOP en data science?**

Menos de lo que crees. Para análisis exploratorio, funciones + dicts/dataclasses bastan. Necesitas clases cuando hay: estado mutable complejo (modelos sklearn), polimorfismo (varios algoritmos misma interfaz), o frameworks que lo exigen (PyTorch nn.Module).

**❓ ¿`@dataclass` o NamedTuple o pydantic?**

NamedTuple: record inmutable simple, sin validación. dataclass: record con métodos opcionales. **pydantic** (no en stdlib): cuando además quieres validación de tipos en runtime, parsing desde JSON, etc. (lo verás en MLOps).

**❓ ¿Composición > herencia siempre?**

Como regla. Usa herencia solo cuando *is-a* sea genuino (`PerroLabrador` is-a `Perro` is-a `Animal`). Para *has-a* (`Coche` tiene un `Motor`), composición. Para reutilizar comportamiento sin jerarquía, considera mixins o protocols.

**❓ ¿`property` y getters/setters Java-style?**

En Python no escribes `getNombre()/setNombre()`. Usa atributo público (`self.nombre = ...`). Si después necesitas lógica, conviertes a `@property` sin cambiar el caller. Es la magia.

**❓ ¿Cuándo `__slots__`?**

Optimización: define los atributos permitidos y ahorra memoria (~50%) al no usar `__dict__` por instancia. Útil solo en clases con millones de instancias. Costo: pierde herencia múltiple y dinamismo.

## 🔗 Referencias

- Ramalho, *Fluent Python* 2e, caps. 5, 11, 14
- [`dataclasses` docs](https://docs.python.org/3/library/dataclasses.html)

➡️ **Siguiente:** [011 — pathlib](../011-pathlib-lectura-y-escritura-de-archivos/README.md)